# 507 — Kong MCP Gateway: Transport Setup & Smoke Test

**Phase 507** puts Kong Serverless in front of the existing hosted MCP server on Railway.
This is transport plumbing only — the app is **not** switched to use Kong for MCP yet.
That switchover happens in Phase 508.

## What this notebook covers

1. What Phase 507 does (and what it deliberately does NOT do)
2. The upstream MCP URL in scope
3. What streamable HTTP is, and why preserving the path matters
4. The three URLs you must not confuse (Konnect API / Kong proxy / upstream MCP)
5. Kong `ai-mcp-proxy` plugin — what it is and why it replaces the plain proxy
6. Step-by-step Konnect UI setup including the `ai-mcp-proxy` plugin
7. decK / reference config examples
8. Environment variable inspection (masked)
9. curl smoke tests
10. Python smoke test (gated by `ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true`)
11. Troubleshooting guide
12. Rollback steps

## What this notebook does NOT cover

- Switching the app's `RemoteMCPToolClient` to route through Kong — **Phase 508**
- Kong consumer ACL hardening — **Phase 508**
- Changes to the Streamlit UI, orchestrator, planner, or trace replay
- Self-managed or hybrid Kong deployments — Konnect Serverless only

---

## Scope boundary summary

| | Phase 507 | Phase 508 |
|---|---|---|
| Kong route `/mcp` created | ✅ | — |
| Smoke tests through Kong | ✅ | — |
| App's `RemoteMCPToolClient` uses Kong | ❌ | ✅ |
| Streamlit UI changed | ❌ | ❌ |
| Planner / orchestrator changed | ❌ | ❌ |
| Existing AI Gateway routes affected | ❌ | ❌ |

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import os
import json
import requests
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

print("Environment loaded.")

Environment loaded.


---

## Part 1 — What Phase 507 does

After Phase 507, the architecture for MCP traffic is:

```
Smoke test / future app client
  ──► Kong Serverless  /mcp  (key-auth validates X-Kong-API-Key)
  ──► https://entity-risk-ai-production.up.railway.app/mcp
```

The upstream Railway MCP server is unchanged. The app still talks directly to
`REMOTE_MCP_URL` in normal operation. Phase 507 only adds the Kong front-door
and verifies traffic flows through it.

---

## Part 2 — Streamable HTTP and why path preservation matters

The MCP server in this project runs over **MCP Streamable HTTP** transport
(as opposed to stdio or SSE).

### What is MCP Streamable HTTP?

MCP Streamable HTTP is the transport used by `FastMCP` when deployed behind a
regular HTTP server (e.g. on Railway).  Clients exchange JSON-RPC messages via
HTTP POST requests and optionally receive streaming responses via Server-Sent
Events (SSE).

The Python MCP SDK's `streamablehttp_client` opens the session by POSTing to
the server URL with a JSON-RPC `initialize` message, then exchanges tool calls
the same way.

### Why path preservation matters

The Railway MCP server listens at exactly **`/mcp`**.  If Kong were configured
with `strip_path: true` (the default), it would strip the `/mcp` prefix before
forwarding — the upstream would receive a bare `/` and return 404.

Setting `strip_path: false` on the Kong route ensures the full `/mcp` path is
forwarded intact:

```
Client sends:    POST https://<kong-proxy>/mcp
Kong forwards:   POST https://entity-risk-ai-production.up.railway.app/mcp  ✅

With strip_path: true (WRONG):
Kong forwards:   POST https://entity-risk-ai-production.up.railway.app/     ❌  404
```

---

## Part 3 — Three URLs you must not confuse

Kong beginners almost always mix these up.  Read this section carefully.

### 1. Konnect API address

```
https://au.api.konghq.com   (or us / eu / in)
```

- Used by **decK** and the **Konnect REST API** to manage configuration.
- Set this as `KONG_KONNECT_ADDR` in your `.env`.
- **This is NOT where you send application or MCP traffic.**
- A common mistake: `KONG_PROXY_URL=https://au.api.konghq.com` — this will not work.

### 2. Serverless proxy URL

```
https://<gateway-id>.au.kong.tech
```

- Where you send real traffic (AI calls, MCP requests, smoke tests).
- Found in Konnect → Gateway Manager → your Serverless gateway.
- Set this as `KONG_PROXY_URL` in your `.env`.
- Appending `/mcp` gives you the Kong MCP route: `KONG_PROXY_URL/mcp`.

### 3. Upstream MCP URL

```
https://entity-risk-ai-production.up.railway.app/mcp
```

- The actual MCP server running on Railway.
- This is what Kong proxies *to* — your app (in Phase 507) never talks here directly.
- Set this as `KONG_MCP_UPSTREAM_URL` in your `.env` (and in the Kong Service config).

| Var | Example value | Purpose |
|---|---|---|
| `KONG_KONNECT_ADDR` | `https://au.api.konghq.com` | decK / Konnect management API |
| `KONG_PROXY_URL` | `https://abc123.au.kong.tech` | Where clients send traffic |
| `KONG_MCP_UPSTREAM_URL` | `https://…railway.app/mcp` | Upstream behind Kong |

---

## Part 4 — Step-by-step Konnect UI setup

Everything in this section is done in the **Konnect Gateway Manager** UI.
No CLI tooling is required for the initial setup.

### Prerequisites

- You have a Konnect Serverless control plane named `entity-risk-ai` (from Phase 505).
- You have the `entity-risk-ai-app` consumer with at least one keyauth credential
  (created in Phase 506).  The existing AI Gateway credential is untouched.
- You know your `KONG_PROXY_URL` (shown in Gateway Manager).

---

### Step 1 — Create the MCP Gateway Service

1. In Konnect, open **Gateway Manager** → select your `entity-risk-ai` control plane.
2. Click **Gateway Services** → **+ New Gateway Service**.
3. Fill in:
   - **Name:** `mcp-upstream-service`
   - **URL:** `https://entity-risk-ai-production.up.railway.app/mcp`
   - Leave timeouts at default (60 000 ms) or increase for slow MCP calls.
4. Click **Save**.

> **Why the full path in the Service URL?**  
> Because the route uses `strip_path: false`, Kong appends the incoming path
> to the service URL as-is.  With the service URL ending in `/mcp` and the
> route path also `/mcp`, requests arrive at the upstream as
> `…railway.app/mcp` — which is correct.

---

### Step 2 — Add the `/mcp` Route

1. Click into `mcp-upstream-service` → **Routes** tab → **+ New Route**.
2. Fill in:
   - **Name:** `mcp-route`
   - **Paths:** `/mcp`
   - **Methods:** `GET`, `POST`
   - **Strip path:** **OFF** ← critical
   - **Protocols:** `HTTPS`
   - **Path handling:** `v0`
3. Click **Save**.

---

### Step 3 — Add the key-auth Plugin to the Route

1. Click into `mcp-route` → **Plugins** tab → **+ Add Plugin**.
2. Choose **key-auth** (under Authentication).
3. Configure:
   - **Key names:** `x-kong-api-key`, `X-Kong-API-Key`
   - **Key in header:** enabled
   - **Key in query:** enabled
   - **Hide credentials:** enabled (strips the key before forwarding)
4. Click **Save**.

---

### Step 4 — Add a Consumer Credential for MCP

1. Go to **Consumers** → click `entity-risk-ai-app`.
2. Click **Credentials** tab → **+ Add Key Auth Credential**.
3. Enter a key value — this becomes your `KONG_MCP_GATEWAY_API_KEY`.
   Use a different value from your AI Gateway key (`KONG_AI_GATEWAY_API_KEY`).
4. Click **Save**.
5. Copy this value into `KONG_MCP_GATEWAY_API_KEY` in your `.env`.

> If you prefer a dedicated consumer for MCP traffic, create a new consumer
> (e.g. `entity-risk-ai-mcp`) and add the credential there instead.

---


---

### Step 4 — Add the `ai-mcp-proxy` Plugin to `mcp-route`

#### What is `ai-mcp-proxy`?

`ai-mcp-proxy` is Kong's native **MCP-protocol-aware** proxy plugin.  It is different from
`ai-proxy-advanced` (used in Phase 506 for Anthropic routing):

| | `ai-proxy-advanced` (Phase 506) | `ai-mcp-proxy` (Phase 507) |
|---|---|---|
| Route | `/ai`, `/ai/sonnet` | `/mcp` |
| Purpose | Routes requests to Anthropic / other LLM providers | Proxies MCP JSON-RPC to an upstream MCP server |
| Upstream | `api.anthropic.com` (injected from plugin config) | Railway FastMCP server |
| Request format | Anthropic Messages API | MCP JSON-RPC (initialize, tools/call, …) |
| LLM traffic? | Yes | No — generic MCP pass-through |

In `passthrough-listener` mode the plugin:
- Understands MCP JSON-RPC at the protocol level (not just forwarding bytes)
- Provides tool-call observability in Konnect → AI Gateway → Analytics
- Lays the foundation for `conversion-listener` mode in Phase 508 (expose REST APIs as MCP tools)

> **License note:** `ai-mcp-proxy` requires the same AI Gateway Enterprise plan as `ai-proxy-advanced`
> (Phase 506).  No new license is needed if Phase 506 already works on your Konnect org.

#### Konnect UI steps

1. Click into `mcp-route` → **Plugins** tab → **+ Add Plugin**.
2. Search for **`AI MCP Proxy`** (plugin name: `ai-mcp-proxy`).
3. Set **Mode** to `passthrough-listener`.
4. Set **max_request_body_size** to `1048576` (1 MB — default).
5. Under **Logging**:
   - `log_audits`: **enabled** ← records which MCP tools were called
   - `log_statistics`: **enabled** ← feeds Konnect AI Gateway analytics
   - `log_payloads`: **disabled** (opt-in — logs full request/response bodies)
6. Leave all other fields at defaults.  In particular:
   - `server.tag` is NOT needed here (only used in `listener` aggregation mode)
   - The upstream MCP server URL stays in the **Service** config, not the plugin
7. Click **Save**.

The route now has two plugins: `key-auth` (runs first) and `ai-mcp-proxy` (runs second).
Kong validates the consumer key before the MCP proxy layer processes the request.

### Step 5 — Populate your `.env`

```dotenv
KONG_PROXY_URL=https://<your-gateway-id>.au.kong.tech
KONG_MCP_GATEWAY_ENABLED=true
KONG_MCP_GATEWAY_ROUTE_PATH=/mcp
KONG_MCP_GATEWAY_API_KEY=<the key you created in Step 4>
KONG_MCP_UPSTREAM_URL=https://entity-risk-ai-production.up.railway.app/mcp
```

Set `ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true` to run the live smoke test cells below.

---

## Part 5 — decK reference examples

The file `kong/declarative/mcp-gateway-reference.yaml` contains a complete
decK reference config for the MCP Gateway.

### Dump the current live state

```bash
deck gateway dump \
  --konnect-addr "$KONG_KONNECT_ADDR" \
  --konnect-token "$KONG_KONNECT_TOKEN" \
  --konnect-control-plane-name "$KONG_KONNECT_CONTROL_PLANE_NAME" \
  --output-file kong/declarative/current-live-state.yaml
```

### Dry-run diff against the reference

```bash
deck gateway diff \
  --konnect-addr "$KONG_KONNECT_ADDR" \
  --konnect-token "$KONG_KONNECT_TOKEN" \
  --konnect-control-plane-name "$KONG_KONNECT_CONTROL_PLANE_NAME" \
  kong/declarative/mcp-gateway-reference.yaml
```

> **Note:** `--konnect-region` is a deprecated flag. Always use `--konnect-addr`.

### `ai-mcp-proxy` plugin — decK reference snippet

Add this plugin alongside `key-auth` on `mcp-route`.  Note plugin ordering:
`key-auth` must appear **before** `ai-mcp-proxy` so unauthenticated requests are
rejected before the MCP proxy layer runs.

```yaml
routes:
- name: mcp-route
  paths:
  - /mcp
  methods:
  - GET
  - POST
  strip_path: false
  plugins:
  - name: key-auth            # 1st — validates X-Kong-API-Key
    config:
      key_names:
      - x-kong-api-key
      - X-Kong-API-Key
      hide_credentials: true
  - name: ai-mcp-proxy        # 2nd — MCP-aware passthrough proxy
    enabled: true
    config:
      mode: passthrough-listener
      max_request_body_size: 1048576
      logging:
        log_audits: true
        log_payloads: false
        log_statistics: true
```

The full reference config (including the service definition) is in
`kong/declarative/mcp-gateway-reference.yaml`.


---

## Part 6 — Environment inspection

In [8]:
from src.config import (
    get_kong_settings,
    get_kong_mcp_gateway_settings,
    get_remote_mcp_url,
)

kong = get_kong_settings()
mcp_gw = get_kong_mcp_gateway_settings()
direct_mcp_url = get_remote_mcp_url()

print("=== Konnect control-plane settings ===")
for k, v in kong.masked().items():
    print(f"  {k}: {v}")

print()
print("=== MCP Gateway settings (masked) ===")
for k, v in mcp_gw.masked().items():
    print(f"  {k}: {v}")

print()
print(f"=== Direct remote MCP URL ===")
print(f"  REMOTE_MCP_URL: {direct_mcp_url or '(not set)'}")

print()
print("=== Live notebook gate ===")
live = os.getenv("ENABLE_LIVE_KONG_NOTEBOOK_TESTS", "").strip().lower() == "true"
print(f"  ENABLE_LIVE_KONG_NOTEBOOK_TESTS: {live}")
if not live:
    print("  → Live smoke test cells are SKIPPED.  Set to true to run them.")

=== Konnect control-plane settings ===
  konnect_region: 
  konnect_addr: https://au.api.konghq.com
  konnect_control_plane_name: entity-risk-ai
  konnect_token: kpat_LuC…***
  konnect_control_plane_id: a92e7da1-469b-4cda-851a-eb326907df7c
  notebook_live_tests: True

=== MCP Gateway settings (masked) ===
  enabled: True
  proxy_url: https://kong-948ef205bdausa0ym.kongcloud.dev
  route_path: /mcp
  api_key: KsxhEYlr…***
  upstream_url: https://entity-risk-ai-production.up.railway.app
  gateway_url: https://kong-948ef205bdausa0ym.kongcloud.dev/mcp

=== Direct remote MCP URL ===
  REMOTE_MCP_URL: https://entity-risk-ai-production.up.railway.app/mcp

=== Live notebook gate ===
  ENABLE_LIVE_KONG_NOTEBOOK_TESTS: True


---

## Part 7 — curl smoke tests

Run these from a terminal after populating your `.env` and completing the
Konnect UI setup above.

Substitute your actual values for `$KONG_PROXY_URL` and `$KONG_MCP_GATEWAY_API_KEY`.

### Why `Accept: application/json, text/event-stream` is required

MCP Streamable HTTP servers validate the `Accept` header on every request.
The server must see **both** `application/json` and `text/event-stream` accepted
by the client, or it rejects the request before processing it with:

```json
{"error":{"code":-32600,"message":"Not Acceptable: Client must accept both application/json and text/event-stream"}}
```

This happens at the MCP transport layer — before any application logic runs.
**Always include this header**, or the request will be rejected by the upstream
regardless of whether your Kong key is correct.

---

### 7.1 — No-auth check (expect 401)

Confirms the route exists and key-auth is enforcing authentication:

```bash
curl -s -o /dev/null -w "%{http_code}" \
  -X POST "$KONG_PROXY_URL/mcp" \
  -H "Accept: application/json, text/event-stream" \
  -H "Content-Type: application/json" \
  -d '{"jsonrpc":"2.0","method":"initialize","id":1,"params":{"protocolVersion":"2024-11-05","capabilities":{},"clientInfo":{"name":"smoke","version":"0.0.1"}}}'
```

**Expected:** `401`
**If 404:** the `/mcp` route does not exist yet — complete Step 2 in Part 4 above.
**If 200:** key-auth is not configured — complete Step 3.
**If 200 with `-32600` error body:** `Accept` header was missing — add it (see above).

---

### 7.2 — Wrong key check (expect 401)

```bash
curl -s -o /dev/null -w "%{http_code}" \
  -X POST "$KONG_PROXY_URL/mcp" \
  -H "Accept: application/json, text/event-stream" \
  -H "X-Kong-API-Key: definitely-wrong-key" \
  -H "Content-Type: application/json" \
  -d '{"jsonrpc":"2.0","method":"initialize","id":1,"params":{"protocolVersion":"2024-11-05","capabilities":{},"clientInfo":{"name":"smoke","version":"0.0.1"}}}'
```

**Expected:** `401`

---

### 7.3 — Valid key, MCP initialize (expect 200 with result body)

```bash
curl -s \
  -X POST "$KONG_PROXY_URL/mcp" \
  -H "Accept: application/json, text/event-stream" \
  -H "X-Kong-API-Key: $KONG_MCP_GATEWAY_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{"jsonrpc":"2.0","method":"initialize","id":1,"params":{"protocolVersion":"2024-11-05","capabilities":{},"clientInfo":{"name":"smoke","version":"0.0.1"}}}'
```

**Expected:** HTTP 200 with a JSON-RPC result body.

**Interpreting responses:**

| HTTP code | Body | Meaning |
|---|---|---|
| `200` | `{"result":...}` | Kong routed successfully; upstream responded |
| `200` | `{"error":{"code":-32600,...}}` | Missing `Accept` header — add it |
| `401` | — | Key not recognised — check key value and consumer credential |
| `403` | — | Key valid but consumer lacks permission |
| `404` | — | Route not found — check `/mcp` route exists in Gateway Manager |
| `502`/`503` | — | Kong cannot reach the Railway upstream |
| `504` | — | Upstream timeout — increase service read_timeout |


### 7.6 — Verify analytics in Konnect (after ai-mcp-proxy is added)

Once the `ai-mcp-proxy` plugin is active with `log_statistics: true`, successful
MCP requests appear in **Konnect → AI Gateway → Analytics** with MCP tool-call metadata.

If you do not see entries after a successful 200 response:
- Confirm `log_statistics: true` in the plugin config
- Wait 1–2 minutes (analytics ingestion has a short delay)
- Check that the plugin is **enabled** on `mcp-route`

---

### 7.4 — Verify key is stripped (hide_credentials)

If your MCP server logs requests, confirm the `X-Kong-API-Key` header does NOT
appear in the upstream request.  Kong strips it when `hide_credentials: true`
is set on the key-auth plugin.

---

### 7.5 — httpie alternative

```bash
http POST "$KONG_PROXY_URL/mcp" \
  "Accept:application/json, text/event-stream" \
  "X-Kong-API-Key:$KONG_MCP_GATEWAY_API_KEY" \
  jsonrpc=2.0 method=initialize id:=1 \
  'params:={"protocolVersion":"2024-11-05","capabilities":{},"clientInfo":{"name":"smoke","version":"0.0.1"}}'
```

---

## Part 8 — Python smoke test

This section tests the Kong MCP Gateway with real HTTP calls.
Cells are gated by `ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true`.

### Test A — no-auth returns 401

In [9]:
live = os.getenv("ENABLE_LIVE_KONG_NOTEBOOK_TESTS", "").strip().lower() == "true"

if not live:
    print("SKIPPED — set ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true to run.")
else:
    mcp_gw = get_kong_mcp_gateway_settings()
    assert mcp_gw.proxy_url, "KONG_PROXY_URL must be set"

    gateway_mcp_url = mcp_gw.gateway_url
    print(f"Testing (no auth): POST {gateway_mcp_url}")

    # The Accept header is required by MCP Streamable HTTP transport.
    # Without it the upstream rejects with -32600 before key-auth fires.
    headers = {
        "Accept": "application/json, text/event-stream",
        "Content-Type": "application/json",
    }
    payload = {
        "jsonrpc": "2.0",
        "method": "initialize",
        "id": 1,
        "params": {
            "protocolVersion": "2024-11-05",
            "capabilities": {},
            "clientInfo": {"name": "smoke-test", "version": "0.0.1"},
        },
    }

    resp = requests.post(gateway_mcp_url, json=payload, headers=headers, timeout=10)
    print(f"  Status: {resp.status_code}")
    assert resp.status_code == 401, (
        f"Expected 401 (key-auth rejection), got {resp.status_code}.\n"
        "  If 200 with -32600 error: Accept header was missing (now fixed).\n"
        "  If plain 200: check that key-auth plugin is attached to mcp-route."
    )
    print("  ✓ 401 returned as expected — key-auth is active.")

Testing (no auth): POST https://kong-948ef205bdausa0ym.kongcloud.dev/mcp
  Status: 401
  ✓ 401 returned as expected — key-auth is active.


### Test B — valid key, MCP initialize

In [10]:
if not live:
    print("SKIPPED — set ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true to run.")
else:
    mcp_gw = get_kong_mcp_gateway_settings()
    assert mcp_gw.proxy_url, "KONG_PROXY_URL must be set"
    assert mcp_gw.api_key, "KONG_MCP_GATEWAY_API_KEY must be set"

    gateway_mcp_url = mcp_gw.gateway_url
    headers = {
        "Accept": "application/json, text/event-stream",
        "Content-Type": "application/json",
        "X-Kong-API-Key": mcp_gw.api_key,
    }
    payload = {
        "jsonrpc": "2.0",
        "method": "initialize",
        "id": 1,
        "params": {
            "protocolVersion": "2024-11-05",
            "capabilities": {},
            "clientInfo": {"name": "smoke-test", "version": "0.0.1"},
        },
    }

    print(f"Testing (valid key): POST {gateway_mcp_url}")
    resp = requests.post(gateway_mcp_url, json=payload, headers=headers, timeout=15)
    print(f"  Status: {resp.status_code}")
    print(f"  Content-Type: {resp.headers.get('content-type', '(none)')}")

    if resp.status_code == 200:
        content_type = resp.headers.get("content-type", "")
        if "application/json" in content_type:
            data = resp.json()
            if "error" in data:
                # -32600 means Accept header is missing — should not happen here
                print(f"  ✗ JSON-RPC error: {data['error']}")
                assert False, "Got JSON-RPC error — check Accept header"
            print(f"  Response: {json.dumps(data, indent=2)[:500]}")
            assert "result" in data, "Response does not look like a JSON-RPC result"
            print("  ✓ JSON-RPC result received through Kong.")
        elif "text/event-stream" in content_type:
            print("  ℹ SSE response — server is in streaming mode.")
            print(f"  First 200 chars: {resp.text[:200]}")
            print("  ✓ SSE stream opened through Kong.")
        else:
            print(f"  Response body (first 200 chars): {resp.text[:200]}")
    else:
        print(f"  ✗ Unexpected status {resp.status_code}: {resp.text[:400]}")
        assert False, f"Expected 200, got {resp.status_code}" 

Testing (valid key): POST https://kong-948ef205bdausa0ym.kongcloud.dev/mcp
  Status: 200
  Content-Type: text/event-stream
  ℹ SSE response — server is in streaming mode.
  First 200 chars: event: message
data: {"jsonrpc":"2.0","id":1,"result":{"protocolVersion":"2024-11-05","capabilities":{"experimental":{},"prompts":{"listChanged":false},"resources":{"subscribe":false,"listChanged":fa
  ✓ SSE stream opened through Kong.


### Test C — MCP SDK initialize via Kong

This test uses the MCP Python SDK's `streamablehttp_client` to open a full
MCP session through Kong.  It exercises the same code path that
`RemoteMCPToolClient` will use in Phase 508.

> **Limitation:** The MCP SDK does not natively support passing custom headers
> (like `X-Kong-API-Key`) in all versions.  If your SDK version does not
> support `headers=` in `streamablehttp_client`, this test will return 401
> from Kong — which is expected behaviour and confirms the key-auth plugin
> is working.  The workaround (Phase 508) is to pass the key via query string
> (`?apikey=...`) or to inject it via a middleware shim.

In [11]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

if not live:
    print("SKIPPED — set ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true to run.")
else:
    mcp_gw = get_kong_mcp_gateway_settings()
    assert mcp_gw.proxy_url, "KONG_PROXY_URL must be set"
    assert mcp_gw.api_key, "KONG_MCP_GATEWAY_API_KEY must be set"

    gateway_mcp_url = mcp_gw.gateway_url
    api_key = mcp_gw.api_key

    print(f"Testing MCP SDK via Kong: {gateway_mcp_url}")
    # Note: the MCP SDK sends Accept: application/json, text/event-stream automatically
    # as part of the MCP Streamable HTTP protocol.  No need to add it manually here.
    # We only need to pass the Kong key-auth header.
    #
    # IPython/Jupyter supports top-level `await` natively (IPython 7+).
    # Do NOT use asyncio.run() here — Jupyter already has a running event loop.
    try:
        async with streamablehttp_client(
            gateway_mcp_url,
            headers={"X-Kong-API-Key": api_key},
        ) as (read, write, _):
            async with ClientSession(read, write) as session:
                result = await session.initialize()
                print(f"  MCP server name   : {result.serverInfo.name}")
                print(f"  MCP server version: {result.serverInfo.version}")
                tools_response = await session.list_tools()
                tool_names = [t.name for t in tools_response.tools]
                print(f"  Available tools ({len(tool_names)}): {tool_names}")
                print("  ✓ Full MCP SDK session opened through Kong successfully.")
    except Exception as exc:
        print(f"  SDK test failed: {type(exc).__name__}: {exc}")
        if "headers" in str(exc).lower() or "unexpected keyword" in str(exc).lower():
            print("  Likely cause: this MCP SDK version does not support headers= in streamablehttp_client.")
            print("  Upgrade mcp[cli] to the latest version.")
        else:
            print("  Kong key-auth is still verified via Test A and Test B above.")

Testing MCP SDK via Kong: https://kong-948ef205bdausa0ym.kongcloud.dev/mcp
  MCP server name   : entity-risk-ai
  MCP server version: 1.27.0


Session termination failed: 404


  Available tools (15): ['resolve_entity', 'validate_plan', 'evaluate_stop_conditions', 'entity_lookup', 'company_profile', 'expand_ownership', 'shared_address_check', 'sic_context', 'ownership_complexity_check', 'control_signal_check', 'address_risk_check', 'industry_context_check', 'retrieve_trace', 'find_traces_by_entity', 'list_recent_traces']
  ✓ Full MCP SDK session opened through Kong successfully.


---

## Part 9 — Direct remote MCP (baseline, unaffected)

Phase 507 does not modify the direct remote MCP path.  This cell confirms it
still works exactly as before.

In [12]:
if not live:
    print("SKIPPED — set ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true to run.")
else:
    direct_url = get_remote_mcp_url()
    if not direct_url:
        print("REMOTE_MCP_URL not set — skipping direct MCP baseline check.")
    else:
        headers = {
            "Accept": "application/json, text/event-stream",
            "Content-Type": "application/json",
        }
        payload = {
            "jsonrpc": "2.0",
            "method": "initialize",
            "id": 1,
            "params": {
                "protocolVersion": "2024-11-05",
                "capabilities": {},
                "clientInfo": {"name": "smoke-baseline", "version": "0.0.1"},
            },
        }
        print(f"Testing direct MCP (no Kong): POST {direct_url}")
        resp = requests.post(direct_url, json=payload, headers=headers, timeout=15)
        print(f"  Status: {resp.status_code}")
        if resp.status_code == 200:
            data = resp.json() if "application/json" in resp.headers.get("content-type","") else {}
            if "result" in data:
                print("  ✓ Direct remote MCP path unaffected by Phase 507.")
            else:
                print(f"  Response: {resp.text[:200]}")
        else:
            print(f"  Note: got {resp.status_code} — check Railway deployment health.")

Testing direct MCP (no Kong): POST https://entity-risk-ai-production.up.railway.app/mcp
  Status: 200
  Response: event: message
data: {"jsonrpc":"2.0","id":1,"result":{"protocolVersion":"2024-11-05","capabilities":{"experimental":{},"prompts":{"listChanged":false},"resources":{"subscribe":false,"listChanged":fa


---

## Part 10 — Troubleshooting

### 200 OK but JSON-RPC error `-32600` — "Not Acceptable"

```json
{"jsonrpc":"2.0","id":"server-error","error":{"code":-32600,"message":"Not Acceptable: Client must accept both application/json and text/event-stream"}}
```

**Cause:** The `Accept` header is missing or incomplete.  MCP Streamable HTTP
servers validate this header at the transport layer before any application logic
runs — including Kong key-auth.  Sending the request without the correct `Accept`
value will always produce this error regardless of whether the Kong key is valid.

**Fix:** Add `-H "Accept: application/json, text/event-stream"` to every curl
command, and `"Accept": "application/json, text/event-stream"` to every
`requests.post()` headers dict.


### `ai-mcp-proxy` plugin not found in Konnect

- The plugin requires **AI Gateway Enterprise** license.
- Sign in to Konnect → click your org name → **Billing** to confirm your plan.
- If your Phase 506 `ai-proxy-advanced` plugin works, your plan is sufficient — the plugin
  may simply not appear if you search for the wrong name.  Search exactly for `ai-mcp-proxy`.

### Requests succeed (200) but no MCP stats appear in Konnect Analytics

- Confirm `log_statistics: true` is set in the `ai-mcp-proxy` plugin config.
- Analytics ingestion has a 1–2 minute delay — wait before re-checking.
- Check that the `ai-mcp-proxy` plugin is **enabled** (toggle in Gateway Manager → `mcp-route` → Plugins).

---

### 401 Unauthorized

- The `X-Kong-API-Key` header value does not match any credential on the consumer.
- Check that `KONG_MCP_GATEWAY_API_KEY` in your `.env` matches exactly what
  was set in Konnect → Consumers → `entity-risk-ai-app` → Credentials.
- Kong key-auth is case-sensitive.

### 403 Forbidden

- The key is recognised but the consumer lacks permission.
- Check that the credential is attached to the correct consumer, and that
  no ACL plugin is blocking the consumer on this route.

### 404 Not Found

- The `/mcp` route has not been created in Konnect yet.
- Or the route path was entered incorrectly (e.g. `mcp` without leading slash).
- Verify in Gateway Manager → `mcp-upstream-service` → Routes.

### 502 Bad Gateway

- Kong could not reach the Railway upstream.
- Check that the Service URL (`https://entity-risk-ai-production.up.railway.app/mcp`)
  is correct and the Railway deployment is running.
- Try `curl https://entity-risk-ai-production.up.railway.app/mcp` directly to
  confirm the upstream is healthy.

### 200 but response body is wrong / strip_path issue

- If `strip_path` is set to **on** (true), Kong strips `/mcp` before forwarding
  and the upstream receives `/`.  Turn strip_path **off** on `mcp-route`.

### MCP SDK test returns 401

- The MCP Python SDK `streamablehttp_client` may not support `headers=` in
  older versions.  Upgrade to the latest MCP SDK or use Test B (raw requests)
  to confirm Kong routing works.  Phase 508 will address the SDK header injection.

### Key-auth not stripping the credential

- Ensure `hide_credentials: true` is set on the key-auth plugin.  Without this,
  the `X-Kong-API-Key` header is forwarded to the Railway upstream.

---

## Part 11 — Rollback

Phase 507 rollback is non-destructive.  The direct remote MCP path is
completely unaffected at all times.

### Option A — disable the env flag (minimal, no Konnect changes)

```dotenv
KONG_MCP_GATEWAY_ENABLED=false
```

This is already the default.  The smoke tests in this notebook will skip.
The app continues using `REMOTE_MCP_URL` directly.

### Option B — disable the route in Konnect (stops all traffic)

1. Konnect → Gateway Manager → `mcp-upstream-service` → Routes.
2. Click `mcp-route` → toggle **Enabled** to **off**.
3. Any request to `$KONG_PROXY_URL/mcp` will now return 404.

### Option C — delete the route and service (full teardown)

1. Delete `mcp-route` from the route list.
2. Delete `mcp-upstream-service` from the service list.
3. Optionally remove the MCP keyauth credential from the consumer.

All Phase 506 AI Gateway routes and consumer credentials are unchanged.

### Confirming the direct path works after rollback

In [13]:
direct_url = get_remote_mcp_url()
if direct_url:
    print(f"Direct MCP URL: {direct_url}")
    print("This path is independent of Kong and always available.")
else:
    print("REMOTE_MCP_URL is not set — set it in .env if you need the direct path.")

Direct MCP URL: https://entity-risk-ai-production.up.railway.app/mcp
This path is independent of Kong and always available.


---

## Part 12 — What is intentionally deferred to Phase 508

Phase 507 is **transport plumbing only**.  The following are out of scope:

| Deferred item | Phase |
|---|---|
| Switch `RemoteMCPToolClient` to use Kong MCP URL | 508 |
| Inject `X-Kong-API-Key` into MCP SDK client | 508 |
| Update Streamlit UI to offer Kong MCP as a transport option | 508 |
| Consumer ACL scopes for MCP vs AI traffic | 508 |
| Rate limiting on the `/mcp` route | 508 |
| Orchestrator / planner changes | 508+ |
| Trace replay changes | 508+ |